# Lab 7: Interactive EDA

## Objectives

By the end of this lab, you should be able to:

- Use Plotly's built-in interaction toolbox (zoom, pan, box and lasso select) as tools for exploring data, not just polish on a finished chart
- Read a scatterplot matrix with linked brushing to spot multivariate patterns and outlier clusters
- Read a parallel coordinates plot and use axis brushing to filter down to a subset of rows
- Use `ipywidgets` to swap which variables drive a chart, without hand-editing code
- Combine widgets and brushing into one exploratory workflow

## Setup

In [2]:
# !pip install pandas plotly ipywidgets nycflights13 palmerpenguins

import pandas as pd
import plotly.express as px
from ipywidgets import interact, widgets
import nycflights13 as nf
from palmerpenguins import load_penguins

penguins = load_penguins().dropna()
flights = nf.flights

## 1. Quick recap

Lab 6 covered the same core EDA questions this lab does: distribution shape, comparisons across categories, and relationships between variables. Everything you learned there about *which chart answers which question* still applies. What changes today is the tool: every chart in this lab is interactive, and interactivity itself becomes part of how you explore, not just how the final chart looks.

## 2. The interaction toolbox

Every Plotly Express chart comes with the same set of built-in tools, available from the toolbar that appears when you hover over a chart: zoom, pan, box select, lasso select, and reset. Let's build one scatterplot and try all of them.

In [ ]:
fig = px.scatter(
    penguins, x="bill_length_mm", y="flipper_length_mm", color="species",
    width=800, height=600,
    hover_data=["island", "body_mass_g"],
    title="Bill length vs. flipper length",
    labels={"bill_length_mm": "Bill length (mm)", "flipper_length_mm": "Flipper length (mm)"}
)
fig.show()

Try each of these on the chart above:

- **Hover** over a point to see every field listed in `hover_data`, not just what's on the axes.
- **Zoom**: click and drag to draw a rectangle; the chart zooms to that region. Double-click to reset.
- **Pan**: switch to the pan tool in the toolbar, then drag to move around without changing the zoom level.
- **Box select** or **lasso select**: switch to one of these tools, then draw around a group of points. Selected points stay full opacity; everything else dims. This is a fast way to isolate a cluster you want to look at more closely, without writing a filter in code.

Selection and zoom are visual tools for you to explore with. They don't change the underlying data or feed into later cells; think of them as a magnifying glass, not a filter step in your analysis.

## 3. Brushing a trend: an interactive line chart

For a real time trend to explore, let's compute the average departure delay per day across all of 2013 in `nycflights13`, the same dataset from Labs 1, 2, and 4.

In [ ]:
daily_delay = (
    flights
    .groupby(["year", "month", "day"], as_index=False)
    .agg(avg_delay=("dep_delay", "mean"))
)
daily_delay["date"] = pd.to_datetime(daily_delay[["year", "month", "day"]])
daily_delay.head()

,year,month,day,avg_delay,date
0,2013,1,1,11.548926,2013-01-01
1,2013,1,2,13.858824,2013-01-02
2,2013,1,3,10.987832,2013-01-03
3,2013,1,4,8.951595,2013-01-04
4,2013,1,5,5.732218,2013-01-05


In [ ]:
# TODO: Create a line chart of average departure delay over time
fig = px.line(

    title="Average departure delay by day, 2013",
    labels={"date": "Date", "avg_delay": "Average departure delay (minutes)"}
)


#TODO: Add a range slider and range selector buttons to the x-axis
# fig.update_xaxes(
    
# )
fig.show()

Click and drag directly on the plot to zoom into a specific stretch of the year, then use the range slider at the bottom to fine-tune what you're looking at. Try zooming into late June through early July, a period commonly associated with summer travel delays. Does the data support that, or does the real spike show up somewhere you didn't expect? This kind of back-and-forth, zoom in, look, zoom back out, try a different range, is exactly the kind of exploration that a static chart can't offer.

## 4. Multivariate patterns with linked brushing

Lab 6 used `sns.pairplot()` for a static, all-at-once overview of several variable pairs. Plotly's equivalent, `px.scatter_matrix()`, adds one important capability for free: **linked brushing**. Select points in any one panel, and the same rows highlight across every other panel simultaneously, with no extra code required.

In [ ]:
splom_vars = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]

#TODO: Create a scatterplot matrix of the penguin measurements, colored by species
fig = px.scatter_matrix(
    
title="Scatterplot matrix of penguin measurements"
)

fig.update_traces(diagonal_visible=False)
fig.update_layout(height=700)
fig.show()

Use box select on any one panel to highlight a group of points, for example the largest-bodied birds in the bottom-right panel. Watch what happens in every other panel: the same individuals light up there too. This is how you'd confirm whether an interesting-looking cluster in one relationship is *also* unusual in other measurements, or whether it's only unusual along the one pair of variables you happened to be looking at.

`diagonal_visible=False` removes the redundant histograms Plotly puts on the diagonal by default; feel free to set it back to `True` if you'd rather see them.

## 5. A new chart type: parallel coordinates

A scatterplot matrix shows every *pair* of variables separately. A **parallel coordinates plot** takes a different approach entirely: each row in the dataset becomes one line, and each variable gets its own vertical axis. A line's path across all the axes at once shows you that row's full profile, not just two variables at a time.

In [ ]:
pc_vars = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]

penguins_pc = penguins.assign(species_code=penguins["species"].astype("category").cat.codes) # adding a numeric code for species to use as color in the parallel coordinates plot  

#TODO: Create a parallel coordinates plot of the penguin measurements, colored by species
fig = px.parallel_coordinates(
    
    title="Parallel coordinates: penguin measurements"
)
fig.show()

Why does this look wrong?

`px.parallel_coordinates` has no support for categorical color, only `color_continuous_scale` on a numeric column. Passing qualitative colors into it makes Plotly treat them as gradient anchors and blend between them, so species aren't cleanly distinguished and the colorbar shows codes instead of names.

Fix: the next cell forces hard-edged discrete color bands instead of a gradient. This a common workaround worth keeping for future parallel coordinates plots.

In [7]:
pc_vars = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]

penguins_pc = penguins.assign(
    species_code=penguins["species"].astype("category").cat.codes
)

species_names = penguins["species"].astype("category").cat.categories
n = len(species_names)
colors = px.colors.qualitative.Plotly[:n]

# build a "discrete-looking" continuous scale: each color repeats
# across its own value band with hard edges instead of blending
discrete_scale = []
for i, color in enumerate(colors):
    discrete_scale.append([i / n, color])
    discrete_scale.append([(i + 1) / n, color])

fig = px.parallel_coordinates(
    penguins_pc, dimensions=pc_vars, color="species_code",
    color_continuous_scale=discrete_scale,
    range_color=[0, n],
    title="Parallel coordinates: penguin measurements"
)

# fix colorbar to show species names at the center of each band, not raw codes
fig.update_layout(
    coloraxis_colorbar=dict(
        title="Species",
        tickvals=[i + 0.5 for i in range(n)],
        ticktext=list(species_names),
    )
)

fig.show()

`parallel_coordinates` colors by a numeric value, so we mapped `species` to a numeric code first with `.cat.codes`; the colorbar on the right won't be as readable as a proper category legend, but the pattern in the lines is still the point here.

Now try **axis brushing**: click and drag along any single axis to select a range on that axis. Every line that passes through your selected range stays highlighted; the rest dim. Try brushing a narrow range on the `flipper_length_mm` axis and watch how those same rows behave on the other three axes. Do the birds with the longest flippers also tend to be heaviest? Do their bill measurements cluster together too, or vary widely?

This is a genuinely different way of asking a multivariate question than the scatterplot matrix. The splom asks "how do these two variables relate," one pair at a time. Parallel coordinates asks "what's this whole row's profile look like," across every variable at once.

## 6. Widget-driven charts

Every chart so far has had its variables hard-coded in the cell. `ipywidgets.interact()` lets you wrap a plotting function so a dropdown, slider, or other control reruns that function automatically whenever the selection changes, without you touching the code.

In [ ]:
numeric_cols = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]

#TODO: use interact to create a scatter plot of the penguin measurements, with x and y axes chosen by the user from the numeric columns
@interact()

def scatter_by_choice(x="bill_length_mm", y="flipper_length_mm"):
    pass

interactive(children=(Dropdown(description='x', options=('bill_length_mm', 'bill_depth_mm', 'flipper_length_mm…

Try every combination of `x` and `y` in the dropdowns above. Which pairs show the clearest separation between species? Which look like there's almost no relationship at all? Notice you didn't write a single new line of plotting code to check each one; the widget reruns the function for you.

## 7. Putting it together

Let's combine a widget with a chart that supports brushing: a multi-select widget that controls which columns feed the parallel coordinates plot, so you can compare different variable combinations without editing code, then brush the result the way you did in Section 5.

In [ ]:
all_numeric = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]

@interact(
    chosen_vars=widgets.SelectMultiple(
        options=all_numeric, value=tuple(all_numeric), description="Variables"
    )
)
def parcoords_by_choice(chosen_vars):
    if len(chosen_vars) < 2:
        print("Select at least two variables.")
        return
    fig = px.parallel_coordinates(
        penguins_pc, dimensions=list(chosen_vars), color="species_code",
        color_continuous_scale=px.colors.qualitative.Plotly[:3],
        title="Parallel coordinates (variables selected above)"
    )
    fig.show()

interactive(children=(SelectMultiple(description='Variables', index=(0, 1, 2, 3), options=('bill_length_mm', '…

Hold Ctrl (or Cmd on Mac) to select multiple variables in the widget above. Try dropping `bill_depth_mm` and comparing the remaining three-axis view to the full four-axis one. Does removing a variable make the pattern easier or harder to read? Then brush an axis on whichever combination you land on, the same way you did in Section 5.

## On your own

- **Task 1:** Using either `penguins` or `flights` dataset, pick one interaction and chart combination from this lab that you have not already used on the same question (for example: widget-driven box plots instead of a scatterplot, or brushing the scatterplot matrix on a pair of variables we didn't discuss above). 

- **Task 2:** In a markdown cell below, write 2-3 sentences on what the interactivity specifically let you notice, something a static version of the same chart would not have shown you as directly.

In [ ]:
# Your code here

*Your written answer here.*

## Submission

Export this notebook to HTML (`Jupyter: Export to HTML` from the Command Palette, or the "..." menu in the notebook toolbar) and submit the HTML file on Canvas.